In [0]:
from pyspark.sql.functions import col, first, broadcast, coalesce, date_trunc, sin, cos, when, concat, lit
import math

print("1. Reading Silver Tables...")
hist_flights = spark.table("aviation_project.silver_historical_flights")
live_flights = spark.table("aviation_project.silver_live_flights")
hist_weather = spark.table("aviation_project.silver_historical_weather")
live_weather = spark.table("aviation_project.silver_live_weather")

In [0]:
# ---------------------------------------------------------
# Part 1: Union the Data Streams
# ---------------------------------------------------------
print("2. Unioning Historical and Live Data...")
# unionByName safely aligns columns even if they are in different orders
all_flights = hist_flights.unionByName(live_flights)
all_weather = hist_weather.unionByName(live_weather)

In [0]:
# ---------------------------------------------------------
# Part 2: The Distance Backfill (Route Dimension)
# ---------------------------------------------------------
print("3. Generating Route Dimension to backfill Live Flight distances...")
route_distances = all_flights.filter(col("distance_miles").isNotNull()) \
    .groupBy("origin_airport", "destination_airport") \
    .agg(first("distance_miles").alias("known_distance"))

flights_filled = all_flights.join(
    broadcast(route_distances), 
    ["origin_airport", "destination_airport"], 
    "left"
).withColumn(
    "distance_miles", coalesce(col("distance_miles"), col("known_distance"))
).drop("known_distance")

In [0]:
# ---------------------------------------------------------
# Part 3: The UTC Time-Series Weather Join
# ---------------------------------------------------------
print("4. Executing Top-of-Hour UTC Weather Joins...")

# Snap both flight and weather UTC timestamps to the top of the hour for matching
flights_filled = flights_filled.withColumn("join_hour_utc", date_trunc("hour", col("scheduled_time_utc")))
all_weather = all_weather.withColumn("join_hour_utc", date_trunc("hour", col("weather_timestamp_utc")))

# Prepare Origin Weather columns
origin_weather = all_weather.withColumnRenamed("airport_code", "origin_airport") \
    .selectExpr("origin_airport", "join_hour_utc", 
                "temperature_f as origin_temp", 
                "wind_speed_mph as origin_wind", 
                "precipitation_inch as origin_precip", 
                "condition as origin_condition")

# Prepare Destination Weather columns
dest_weather = all_weather.withColumnRenamed("airport_code", "destination_airport") \
    .selectExpr("destination_airport", "join_hour_utc", 
                "temperature_f as dest_temp", 
                "wind_speed_mph as dest_wind", 
                "precipitation_inch as dest_precip", 
                "condition as dest_condition")

# Perform the double-join
gold_df = flights_filled \
    .join(origin_weather, ["origin_airport", "join_hour_utc"], "left") \
    .join(dest_weather, ["destination_airport", "join_hour_utc"], "left")

In [0]:
# ---------------------------------------------------------
# Part 4: Advanced Feature Engineering
# ---------------------------------------------------------
print("5. Engineering Cyclical Time and Holiday Features...")

# 1. Cyclical Time Encoding
gold_df = gold_df.withColumn("minute_of_day", col("hour") * 60 + col("minute"))
gold_df = gold_df.withColumn("time_sin", sin(col("minute_of_day") * (2 * math.pi / 1440))) \
                 .withColumn("time_cos", cos(col("minute_of_day") * (2 * math.pi / 1440)))

# 2. Holiday Proximity Indicator
gold_df = gold_df.withColumn(
    "is_holiday",
    when((col("month") == 12) & (col("day_of_month") >= 20), 1)
    .when((col("month") == 1) & (col("day_of_month") <= 4), 1)
    .when((col("month") == 11) & (col("day_of_month").between(20, 29)), 1)
    .when((col("month") == 7) & (col("day_of_month").between(3, 6)), 1)
    .otherwise(0)
)

# 3. Explicit Route Feature 
gold_df = gold_df.withColumn("route", concat(col("origin_airport"), lit("-"), col("destination_airport")))

# Drop columns the ML model no longer needs
gold_df = gold_df.drop("join_hour_utc", "minute_of_day")

# Drop any flights that are missing weather data
gold_df = gold_df.dropna(subset=["origin_temp", "dest_temp"])

In [0]:
display(gold_df.limit(5))

In [0]:
print(f"3. Gold Master Dataset contains {gold_df.count()} rows.")

In [0]:
# ---------------------------------------------------------
# Part 5: Write to Gold
# ---------------------------------------------------------
print("6. Writing Master Dataset to Gold Delta Table...")
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("aviation_project.gold_master_dataset")

spark.sql("OPTIMIZE aviation_project.gold_master_dataset ZORDER BY (scheduled_time_utc)")

final_count = spark.table("aviation_project.gold_master_dataset").count()
print(f"7. SUCCESS! Gold Master Dataset generated with {final_count} ML-ready rows.")